# RSNA Knee Abnormality Detection — EDA Phase 2

This notebook continues the dataset audit after the first single-study DICOM exploration.

## Main questions

1. How many series does each knee study contain?
2. Which MRI plane / fluid-sensitive protocol combinations are common?
3. Are the 58 gold-labeled studies representative of the other 4,349 studies?
4. What scanner manufacturers, field strengths, resolutions, spacings and slice counts occur?
5. Are there train/test protocol differences?
6. What do the multilingual radiology reports look like at dataset scale?
7. Are there duplicate or highly templated reports?
8. How do the 12 gold labels co-occur?
9. Which report phrases are associated with each gold target?
10. What outliers or QC problems must be handled before modeling?

This notebook does **EDA only**. It does not create final pseudo-labels and it does not train the competition model.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import os, re, math, json, warnings, unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

SEED = 2026
rng = np.random.default_rng(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("Imports ready.")

## 1. Locate and load the competition data

In [ ]:
CANDIDATES = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
]

DATA_DIR = next((p for p in CANDIDATES if p.exists()), None)

if DATA_DIR is None:
    matches = list(Path("/kaggle/input").rglob("rsna-knee-abnormality-detection"))
    DATA_DIR = matches[0] if matches else None

if DATA_DIR is None:
    raise FileNotFoundError("Competition dataset not found.")

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")

train["n_labels_available"] = train[TARGETS].notna().sum(axis=1)
train["is_gold"] = train["n_labels_available"].eq(len(TARGETS))

print("DATA_DIR:", DATA_DIR)
print("Train studies:", len(train))
print("Train series:", len(train_series))
print("Visible test studies:", len(test))
print("Visible test series:", len(test_series))
print("Gold studies:", train["is_gold"].sum())

## 2. Study-level series counts

This measures how many MRI acquisitions are present per examination.

In [ ]:
train_series_count = train_series.groupby("StudyInstanceUID").size().rename("n_series")
test_series_count = test_series.groupby("StudyInstanceUID").size().rename("n_series")

display(train_series_count.describe().to_frame("train"))
display(train_series_count.value_counts().sort_index().to_frame("number_of_studies"))

plt.figure(figsize=(9,4))
plt.hist(train_series_count.values, bins=np.arange(train_series_count.min()-0.5, train_series_count.max()+1.5, 1))
plt.xlabel("MRI series per study")
plt.ylabel("Number of studies")
plt.title("Training study series-count distribution")
plt.show()

## 3. MRI protocol distribution

The organizer metadata gives:
- `Anatomical_Plane`
- `Fluid_Sensitive`
- `Fat_Suppression`

We first check how these variables behave globally.

In [ ]:
for col in ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]:
    print("\n", col)
    display(train_series[col].value_counts(dropna=False).to_frame("count"))

combo = (
    train_series
    .groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)
display(combo)

agreement = (train_series["Fluid_Sensitive"] == train_series["Fat_Suppression"]).mean()
print(f"Fluid_Sensitive == Fat_Suppression in {agreement:.2%} of training series")

## 4. Protocol fingerprint per study

For each knee, summarize which plane/sequence combinations exist.

This is important because the model should not assume every study contains exactly the same protocol.

In [ ]:
def study_protocol_features(series_df):
    x = series_df.copy()
    x["slot"] = (
        x["Anatomical_Plane"].astype(str) +
        "_F" + x["Fluid_Sensitive"].astype(str) +
        "_FS" + x["Fat_Suppression"].astype(str)
    )

    base = x.groupby("StudyInstanceUID").agg(
        n_series=("SeriesInstanceUID", "size"),
        n_planes=("Anatomical_Plane", "nunique"),
        n_fluid=("Fluid_Sensitive", "sum"),
        n_fatsupp=("Fat_Suppression", "sum"),
    )

    for plane in ["Sagittal", "Coronal", "Axial"]:
        base[f"n_{plane.lower()}"] = (
            x["Anatomical_Plane"].eq(plane)
            .groupby(x["StudyInstanceUID"])
            .sum()
        )

    slot_counts = pd.crosstab(x["StudyInstanceUID"], x["slot"])
    base = base.join(slot_counts, how="left").fillna(0)

    return base

train_protocol = study_protocol_features(train_series)
test_protocol = study_protocol_features(test_series)

display(train_protocol.head())
print("\nNumber of unique protocol fingerprints:",
      train_protocol.drop(columns=[]).astype(str).agg("|".join, axis=1).nunique())

## 5. Most common study protocols

In [ ]:
protocol_cols = [
    "n_sagittal", "n_coronal", "n_axial",
    "n_fluid", "n_fatsupp", "n_series"
]

protocol_counts = (
    train_protocol[protocol_cols]
    .value_counts()
    .rename("studies")
    .reset_index()
)

display(protocol_counts.head(20))

## 6. Gold subset representativeness — protocol

The 58 gold cases are precious for evaluation, but we need to know whether they look like a random subset of the full training set.

If they differ strongly in protocol, language, report length, scanner, etc., we should **not** treat gold performance as a simple estimate of overall population performance.

In [ ]:
study_base = train[["StudyInstanceUID", "Report", "is_gold"]].copy()
study_base = study_base.merge(
    train_protocol.reset_index(),
    on="StudyInstanceUID",
    how="left"
)

summary = (
    study_base
    .groupby("is_gold")[["n_series","n_planes","n_fluid","n_fatsupp",
                         "n_sagittal","n_coronal","n_axial"]]
    .agg(["mean","std","median"])
)
display(summary)

## 7. Gold label prevalence and label cardinality

Because this is multi-label classification, one study can contain several positive findings.

In [ ]:
gold = train[train["is_gold"]].copy()

label_summary = pd.DataFrame({
    "positive": gold[TARGETS].sum(),
    "negative": (1 - gold[TARGETS]).sum(),
})
label_summary["positive_pct"] = 100 * label_summary["positive"] / len(gold)
display(label_summary.sort_values("positive_pct", ascending=False))

gold["n_positive_labels"] = gold[TARGETS].sum(axis=1)

display(gold["n_positive_labels"].describe().to_frame())
display(gold["n_positive_labels"].value_counts().sort_index().to_frame("studies"))

plt.figure(figsize=(8,4))
plt.hist(gold["n_positive_labels"], bins=np.arange(-0.5, gold["n_positive_labels"].max()+1.5, 1))
plt.xlabel("Positive abnormalities per gold study")
plt.ylabel("Number of gold studies")
plt.title("Gold-set label cardinality")
plt.show()

## 8. Gold label co-occurrence

This shows which findings tend to occur in the same knee.

Because only 58 gold studies exist, treat this as exploratory rather than a stable epidemiological estimate.

In [ ]:
cooccur = gold[TARGETS].T.dot(gold[TARGETS]).astype(int)
display(cooccur)

corr = gold[TARGETS].corr()

plt.figure(figsize=(10,9))
im = plt.imshow(corr.values, vmin=-1, vmax=1)
plt.xticks(range(len(TARGETS)), TARGETS, rotation=90)
plt.yticks(range(len(TARGETS)), TARGETS)
plt.colorbar(im, label="Pearson correlation")
plt.title("Gold-label correlation matrix")
plt.tight_layout()
plt.show()

# 9. Report EDA

All 4,407 training studies have reports.

We characterize:
- length,
- line count,
- likely writing system,
- repeated text,
- potential language groups,
- gold vs unlabeled report differences.

In [ ]:
train["report_chars"] = train["Report"].fillna("").str.len()
train["report_words"] = train["Report"].fillna("").str.split().str.len()
train["report_lines"] = train["Report"].fillna("").str.count("\n") + 1

display(train[["report_chars","report_words","report_lines"]].describe())

by_gold = train.groupby("is_gold")[["report_chars","report_words","report_lines"]].agg(["mean","median","std"])
display(by_gold)

plt.figure(figsize=(9,4))
plt.hist(train.loc[~train["is_gold"], "report_chars"], bins=50, alpha=0.6, label="unlabeled")
plt.hist(train.loc[train["is_gold"], "report_chars"], bins=20, alpha=0.7, label="gold")
plt.xlabel("Report length (characters)")
plt.ylabel("Studies")
plt.legend()
plt.title("Report-length distribution")
plt.show()

## 10. Lightweight multilingual report grouping

This is **EDA only**, not the final NLP labeler.

The detector uses scripts and common function words. It is designed only to reveal major language groups without requiring internet access or extra packages.

In [ ]:
LANG_HINTS = {
    "English": [" the ", " no ", " knee ", " ligament ", " meniscus ", " findings ", " impression "],
    "Spanish": [" no hay ", " rodilla ", " menisco ", " ligamento ", " hallazgos ", " impresión ", " derrame "],
    "French": [" genou ", " menisque ", " ligament ", " aucune ", " pas de ", " constatations ", " épanchement "],
    "German/Dutch": [" knie ", " menisk", " kreuzband", " geen ", " zonder ", " bevinding", " keine ", " befund"],
    "Turkish": [" diz ", " menisk", " bağ", " izlen", " saptan", " eklem ", " yoktur "],
    "Balkan-Latin": [" koljen", " kolen", " menisk", " ligament", " nema ", " bez "],
}

def script_group(text):
    counts = Counter()
    for ch in text:
        if "\\u0370" <= ch <= "\\u03ff":
            counts["Greek"] += 1
        elif "\\u0400" <= ch <= "\\u04ff":
            counts["Cyrillic"] += 1
        elif ch.isalpha():
            counts["Latin"] += 1
    if counts["Greek"] > 10:
        return "Greek"
    if counts["Cyrillic"] > 10:
        return "Cyrillic"
    return "Latin"

def rough_language(text):
    sg = script_group(text)
    if sg != "Latin":
        return sg

    t = " " + text.casefold() + " "
    scores = {
        lang: sum(t.count(h.casefold()) for h in hints)
        for lang, hints in LANG_HINTS.items()
    }
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Latin-Other"

train["rough_language"] = train["Report"].map(rough_language)

display(train["rough_language"].value_counts().to_frame("studies"))

lang_gold = pd.crosstab(train["rough_language"], train["is_gold"])
lang_gold.columns = ["unlabeled", "gold"] if len(lang_gold.columns)==2 else lang_gold.columns
display(lang_gold)

## 11. Duplicate / templated report audit

Exact duplicates can reveal copied templates or repeated reports.

This matters because random study-level CV may still contain highly similar report templates across folds, making NLP validation look easier than true generalization.

In [ ]:
norm_report = (
    train["Report"]
    .fillna("")
    .str.casefold()
    .str.replace(r"\[date\]|\[time\]", " ", regex=True)
    .str.replace(r"\d+", "0", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

train["report_norm"] = norm_report
dup_counts = train["report_norm"].value_counts()

print("Exact normalized duplicate groups:", int((dup_counts > 1).sum()))
print("Studies belonging to duplicate groups:", int(dup_counts[dup_counts > 1].sum()))

display(dup_counts.head(20).to_frame("count"))

# 12. Gold report phrase exploration

For each target, show snippets from positive and negative gold reports.

This is one of the most important bridges between EDA and our later pseudo-labeling system.

In [ ]:
TARGET_TERMS = {
    "ACL": ["acl", "anterior cruciate", "cruzado anterior", "kreuzband"],
    "MCL": ["mcl", "medial collateral", "colateral medial"],
    "Medial Meniscus": ["medial meniscus", "menisco medial", "menisco interno"],
    "Lateral Meniscus": ["lateral meniscus", "menisco lateral", "menisco externo"],
    "Medial OA": ["medial compartment", "medial femorotibial", "femorotibial medial"],
    "Lateral OA": ["lateral compartment", "lateral femorotibial", "femorotibial lateral"],
    "PF OA": ["patellofemoral", "patellar", "trochlea", "rótula", "rotula"],
    "Effusion": ["effusion", "derrame", "épanchement"],
    "Synovitis": ["synovitis", "sinovitis", "synoviale"],
    "Baker's": ["baker", "popliteal", "quiste poplite"],
    "Contusion": ["contusion", "bone marrow edema", "marrow edema", "edema óseo"],
    "Fracture": ["fracture", "fractura", "fraktur"],
}

def best_snippet(report, terms, radius=180):
    t = report.casefold()
    pos = None
    for term in terms:
        p = t.find(term.casefold())
        if p >= 0 and (pos is None or p < pos):
            pos = p
    if pos is None:
        return report[:2*radius].replace("\\n"," ")
    lo = max(0, pos-radius)
    hi = min(len(report), pos+radius)
    return report[lo:hi].replace("\\n"," ")

for target in TARGETS:
    print("\\n" + "="*110)
    print(target)
    print("="*110)

    pos = gold[gold[target] == 1].head(3)
    neg = gold[gold[target] == 0].head(3)

    print("\\nPOSITIVE GOLD EXAMPLES")
    for _, r in pos.iterrows():
        print("-", best_snippet(r["Report"], TARGET_TERMS[target]))

    print("\\nNEGATIVE GOLD EXAMPLES")
    for _, r in neg.iterrows():
        print("-", best_snippet(r["Report"], TARGET_TERMS[target]))

# 13. Dataset-wide DICOM inventory

This scans **one header per MRI series** plus folder file counts.

It is much cheaper than decoding every image, while still giving the key acquisition/QC variables.

Expected runtime depends on Kaggle storage speed.

The result is cached to `/kaggle/working/rsna_series_inventory.csv`.

In [ ]:
INVENTORY_PATH = Path("/kaggle/working/rsna_series_inventory.csv")

def first_file_in_dir(path):
    try:
        for p in path.iterdir():
            if p.is_file():
                return p
    except Exception:
        return None
    return None

def to_float_list(x):
    try:
        return [float(v) for v in x]
    except Exception:
        return None

def inspect_series(record, split="train"):
    study = str(record.StudyInstanceUID)
    series = str(record.SeriesInstanceUID)

    root = DATA_DIR / f"{split}_series" / study / series

    try:
        files = [p for p in root.iterdir() if p.is_file()]
    except Exception as e:
        return {
            "split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
            "error": f"folder:{type(e).__name__}"
        }

    if not files:
        return {
            "split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
            "n_files": 0, "error": "empty"
        }

    f = files[0]

    try:
        ds = pydicom.dcmread(str(f), stop_before_pixels=True)
    except Exception as e:
        return {
            "split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
            "n_files": len(files), "error": f"dicom:{type(e).__name__}"
        }

    ps = to_float_list(getattr(ds, "PixelSpacing", None))

    return {
        "split": split,
        "StudyInstanceUID": study,
        "SeriesInstanceUID": series,
        "Anatomical_Plane": record.Anatomical_Plane,
        "Fluid_Sensitive": int(record.Fluid_Sensitive),
        "Fat_Suppression": int(record.Fat_Suppression),
        "n_files": len(files),
        "Rows": getattr(ds, "Rows", np.nan),
        "Columns": getattr(ds, "Columns", np.nan),
        "PixelSpacingRow": ps[0] if ps else np.nan,
        "PixelSpacingCol": ps[1] if ps else np.nan,
        "SliceThickness": float(getattr(ds, "SliceThickness", np.nan)),
        "SpacingBetweenSlices": float(getattr(ds, "SpacingBetweenSlices", np.nan)),
        "Manufacturer": str(getattr(ds, "Manufacturer", "")),
        "ManufacturerModelName": str(getattr(ds, "ManufacturerModelName", "")),
        "MagneticFieldStrength": float(getattr(ds, "MagneticFieldStrength", np.nan)),
        "RepetitionTime": float(getattr(ds, "RepetitionTime", np.nan)),
        "EchoTime": float(getattr(ds, "EchoTime", np.nan)),
        "FlipAngle": float(getattr(ds, "FlipAngle", np.nan)),
        "PhotometricInterpretation": str(getattr(ds, "PhotometricInterpretation", "")),
        "BitsStored": getattr(ds, "BitsStored", np.nan),
        "PixelRepresentation": getattr(ds, "PixelRepresentation", np.nan),
        "error": "",
    }

if INVENTORY_PATH.exists():
    inventory = pd.read_csv(INVENTORY_PATH)
    print("Loaded cached inventory:", INVENTORY_PATH)
else:
    tasks = []
    with ThreadPoolExecutor(max_workers=12) as ex:
        for r in train_series.itertuples(index=False):
            tasks.append(ex.submit(inspect_series, r, "train"))
        for r in test_series.itertuples(index=False):
            tasks.append(ex.submit(inspect_series, r, "test"))

        rows = []
        for i, fut in enumerate(as_completed(tasks), 1):
            rows.append(fut.result())
            if i % 2500 == 0:
                print(f"Inspected {i:,}/{len(tasks):,} series")

    inventory = pd.DataFrame(rows)
    inventory.to_csv(INVENTORY_PATH, index=False)
    print("Saved:", INVENTORY_PATH)

display(inventory.head())
print("Rows:", len(inventory))
print("Errors:", (inventory["error"].fillna("") != "").sum())

## 14. Slice-count distribution

In [ ]:
train_inv = inventory[inventory["split"] == "train"].copy()
test_inv = inventory[inventory["split"] == "test"].copy()

display(train_inv["n_files"].describe().to_frame("slices_per_series"))

for plane in ["Sagittal", "Coronal", "Axial"]:
    x = train_inv.loc[train_inv["Anatomical_Plane"] == plane, "n_files"].dropna()
    print("\n", plane)
    display(x.describe().to_frame())

plt.figure(figsize=(9,4))
plt.hist(train_inv["n_files"].dropna(), bins=60)
plt.xlabel("Slices per MRI series")
plt.ylabel("Series")
plt.title("Dataset-wide slice-count distribution")
plt.show()

## 15. Image matrix and spacing distributions

In [ ]:
for col in ["Rows","Columns","PixelSpacingRow","SliceThickness","SpacingBetweenSlices"]:
    print("\n", col)
    display(train_inv[col].describe().to_frame())
    display(train_inv[col].value_counts(dropna=False).head(15).to_frame("count"))

## 16. Scanner manufacturer / field strength

In [ ]:
display(train_inv["Manufacturer"].replace("", np.nan).value_counts(dropna=False).to_frame("series"))
display(train_inv["MagneticFieldStrength"].value_counts(dropna=False).sort_index().to_frame("series"))

scanner_summary = (
    train_inv
    .groupby(["Manufacturer","MagneticFieldStrength"], dropna=False)
    .size()
    .rename("series")
    .reset_index()
    .sort_values("series", ascending=False)
)
display(scanner_summary.head(30))

## 17. Gold vs unlabeled scanner comparison

In [ ]:
study_scanner = (
    train_inv
    .groupby("StudyInstanceUID")
    .agg(
        manufacturer=("Manufacturer", lambda s: s.mode().iloc[0] if len(s.mode()) else ""),
        field_strength=("MagneticFieldStrength", "median"),
        median_slices=("n_files", "median"),
        max_slices=("n_files", "max"),
        median_px=("PixelSpacingRow", "median"),
        median_thickness=("SliceThickness", "median"),
    )
    .reset_index()
)

gold_compare = (
    train[["StudyInstanceUID","is_gold","report_chars","report_words","rough_language"]]
    .merge(study_scanner, on="StudyInstanceUID", how="left")
)

print("Manufacturer × gold:")
display(pd.crosstab(gold_compare["manufacturer"], gold_compare["is_gold"]))

print("\nField strength × gold:")
display(pd.crosstab(gold_compare["field_strength"], gold_compare["is_gold"]))

## 18. Train vs visible-test acquisition comparison

The visible test contains only 3 studies, so do not overinterpret percentages.

The point is to make sure inference code handles the observed test geometry dynamically.

In [ ]:
compare_cols = [
    "n_files","Rows","Columns","PixelSpacingRow",
    "SliceThickness","SpacingBetweenSlices","MagneticFieldStrength"
]

summary_rows = []
for col in compare_cols:
    for split, df in [("train", train_inv), ("test", test_inv)]:
        x = pd.to_numeric(df[col], errors="coerce").dropna()
        summary_rows.append({
            "feature": col,
            "split": split,
            "n": len(x),
            "mean": x.mean() if len(x) else np.nan,
            "median": x.median() if len(x) else np.nan,
            "min": x.min() if len(x) else np.nan,
            "max": x.max() if len(x) else np.nan,
        })

display(pd.DataFrame(summary_rows))

# 19. Sampled intensity audit

MRI intensity is not standardized like CT Hounsfield Units.

We therefore inspect a random subset of series before deciding the training normalization strategy.

This cell decodes one middle slice from a random sample of up to 500 training series.

In [ ]:
N_INTENSITY_SERIES = min(500, len(train_inv))
sample_inv = train_inv.sample(N_INTENSITY_SERIES, random_state=SEED)

def middle_file_sorted(folder):
    files = [p for p in folder.iterdir() if p.is_file()]
    if not files:
        return None

    def key(p):
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
            inst = getattr(ds, "InstanceNumber", None)
            return float(inst) if inst is not None else p.name
        except Exception:
            return p.name

    files = sorted(files, key=key)
    return files[len(files)//2]

intensity_rows = []

for i, r in sample_inv.iterrows():
    folder = DATA_DIR / "train_series" / str(r["StudyInstanceUID"]) / str(r["SeriesInstanceUID"])
    f = middle_file_sorted(folder)

    if f is None:
        continue

    try:
        ds = pydicom.dcmread(str(f))
        img = ds.pixel_array.astype(np.float32)

        slope = float(getattr(ds, "RescaleSlope", 1.0))
        intercept = float(getattr(ds, "RescaleIntercept", 0.0))
        img = img * slope + intercept

        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = img.max() - img

        intensity_rows.append({
            "StudyInstanceUID": r["StudyInstanceUID"],
            "SeriesInstanceUID": r["SeriesInstanceUID"],
            "Plane": r["Anatomical_Plane"],
            "Fluid": r["Fluid_Sensitive"],
            "P1": np.percentile(img, 1),
            "P50": np.percentile(img, 50),
            "P99": np.percentile(img, 99),
            "Mean": img.mean(),
            "Std": img.std(),
        })
    except Exception:
        pass

intensity_df = pd.DataFrame(intensity_rows)
display(intensity_df.describe(include="all"))

plt.figure(figsize=(9,4))
plt.hist(intensity_df["P99"] - intensity_df["P1"], bins=50)
plt.xlabel("P99 - P1 intensity range")
plt.ylabel("Sampled series")
plt.title("MRI intensity-range variability")
plt.show()

# 20. Outlier tables

These are the studies/series we should manually inspect before training.

In [ ]:
print("Very low slice count:")
display(train_inv.nsmallest(20, "n_files")[
    ["StudyInstanceUID","SeriesInstanceUID","Anatomical_Plane","n_files","Rows","Columns"]
])

print("\nVery high slice count:")
display(train_inv.nlargest(20, "n_files")[
    ["StudyInstanceUID","SeriesInstanceUID","Anatomical_Plane","n_files","Rows","Columns"]
])

print("\nLargest slice thickness:")
display(train_inv.nlargest(20, "SliceThickness")[
    ["StudyInstanceUID","SeriesInstanceUID","Anatomical_Plane","SliceThickness","SpacingBetweenSlices"]
])

# 21. Build a study-level EDA feature table

This table will become useful later for:
- fold design,
- subgroup checks,
- scanner/site robustness,
- gold-set representativeness,
- failure analysis.

In [ ]:
study_eda = (
    train[[
        "StudyInstanceUID","is_gold","Report",
        "report_chars","report_words","report_lines","rough_language"
    ]]
    .merge(train_protocol.reset_index(), on="StudyInstanceUID", how="left")
    .merge(study_scanner, on="StudyInstanceUID", how="left")
)

for t in TARGETS:
    study_eda[t] = train.set_index("StudyInstanceUID").loc[
        study_eda["StudyInstanceUID"], t
    ].values

OUT_STUDY = Path("/kaggle/working/rsna_study_eda_features.csv")
study_eda.drop(columns=["Report"]).to_csv(OUT_STUDY, index=False)

print("Saved:", OUT_STUDY)
display(study_eda.head())

# 22. EDA decision checklist

After this notebook finishes, record the following:

### MRI acquisition
- Typical and extreme number of series per study
- Typical and extreme slices per series
- Dominant protocol fingerprints
- Whether all three planes are usually available
- Whether fluid-sensitive and fat-suppression flags are redundant
- Scanner manufacturers and 1.5T / 3T distribution
- Resolution / spacing / thickness variability
- Outlier series requiring exclusion or special handling

### Reports
- Approximate language distribution
- Gold vs unlabeled language differences
- Report length differences
- Duplicate/template frequency
- Examples where textual findings do not map trivially to challenge labels

### Gold labels
- Number of positive labels per case
- Label correlations / co-occurrence
- Whether the 58 cases are protocol/language/scanner representative

### Modeling consequences
Only after these are known should we freeze:
- fold strategy,
- series-slot strategy,
- slice sampling,
- intensity normalization,
- pseudo-labeling method,
- first MRI baseline.

# 23. Files to keep

After running the notebook, download or preserve:

- `rsna_series_inventory.csv`
- `rsna_study_eda_features.csv`

When you send me the executed notebook or those two CSVs, we can make the next decisions quantitatively rather than by guesswork.